In [32]:
pip install pandas


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [33]:
import pandas as pd

In [34]:
df = pd.read_csv("results.csv")
df

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False
...,...,...,...,...,...,...,...,...,...
49500,2026-07-07,Switzerland,Colombia,0.0,0.0,FIFA World Cup,Vancouver,Canada,True
49501,2026-07-09,France,Morocco,NaN,NaN,FIFA World Cup,Foxborough,United States,True
49502,2026-07-10,Spain,Belgium,NaN,NaN,FIFA World Cup,Inglewood,United States,True
49503,2026-07-11,Norway,England,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True


In [35]:
df["date"] = pd.to_datetime(df["date"], format = '%Y-%m-%d')
df.sort_values(by="date")
df

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False
...,...,...,...,...,...,...,...,...,...
49500,2026-07-07,Switzerland,Colombia,0.0,0.0,FIFA World Cup,Vancouver,Canada,True
49501,2026-07-09,France,Morocco,NaN,NaN,FIFA World Cup,Foxborough,United States,True
49502,2026-07-10,Spain,Belgium,NaN,NaN,FIFA World Cup,Inglewood,United States,True
49503,2026-07-11,Norway,England,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True


In [36]:
elo_record = {}

In [37]:
def expectedVals(teamA_elo, teamB_elo):
    # Expected score for teamA and teamB
    expA = 1/(1 + 10**((teamB_elo-teamA_elo)/400))
    expB = 1 - expA
    return expA, expB

In [38]:
def actualRating(teamA_score, teamB_score):
    # what to do if there are rows with no scores, maybe just delete the rows that don't have score in them
    if teamA_score > teamB_score:
        return  1, 0
    if teamA_score < teamB_score:
        return 0, 1
    else:
        return 0.5, 0.5



In [39]:
def updatedRating(teamA_elo, teamB_elo, teamA_score, teamB_score, k_factor):
    expA, expB = expectedVals(teamA_elo, teamB_elo)
    actA, actB = actualRating(teamA_score, teamB_score)
    newA = teamA_elo + k_factor * (actA - expA)
    newB = teamB_elo + k_factor * (actB - expB)
    return newA, newB

In [ ]:
for index, row in df.iterrows():
    # if the current row teams doesnt have an elo in elo_record intialize it to the value of 1500
    if row["home_team"] not in elo_record:
        elo_record[row["home_team"]] = 1500
    if row["away_team"] not in elo_record:
        elo_record[row["away_team"]] = 1500
    # now add 2 new columns for each row with the current elo_record
    df.loc[index, "home_elo"] = elo_record[row["home_team"]]
    df.loc[index, "away_elo"] = elo_record[row["away_team"]]
    # now update the elo_record given the score of the game
    home_elo = elo_record[row["home_team"]]
    away_elo = elo_record[row["away_team"]]
    home_score = row["home_score"]
    away_score = row["away_score"]
    newHome, newAway = updatedRating(home_elo, away_elo, home_score, away_score, k_factor = 20)
    elo_record[row["home_team"]] = newHome
    elo_record[row["away_team"]] = newAway

   


In [41]:
df

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo,away_elo
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1500.000000,1500.000000
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,1500.000000,1500.000000
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1484.000000,1516.000000
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,1498.530498,1501.469502
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1501.334159,1498.665841
...,...,...,...,...,...,...,...,...,...,...,...
49500,2026-07-07,Switzerland,Colombia,0.0,0.0,FIFA World Cup,Vancouver,Canada,True,1882.747143,1955.092799
49501,2026-07-09,France,Morocco,NaN,NaN,FIFA World Cup,Foxborough,United States,True,2031.828390,1941.478871
49502,2026-07-10,Spain,Belgium,NaN,NaN,FIFA World Cup,Inglewood,United States,True,2072.564084,1911.222769
49503,2026-07-11,Norway,England,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True,1897.368670,1978.024720


In [ ]:
for country in elo_record:
    print(f"Country: {country} Elo: {elo_record[country]}")

In [67]:
home = df[["date", "home_team", "away_team", "home_score", "away_score"]].copy()
home.columns = ["date", "team", "opponent", "goals_scored", "opponent_score"]
home

,date,team,opponent,goals_scored,opponent_score
0,1872-11-30,Scotland,England,0.0,0.0
1,1873-03-08,England,Scotland,4.0,2.0
2,1874-03-07,Scotland,England,2.0,1.0
3,1875-03-06,England,Scotland,2.0,2.0
4,1876-03-04,Scotland,England,3.0,0.0
...,...,...,...,...,...
49500,2026-07-07,Switzerland,Colombia,0.0,0.0
49501,2026-07-09,France,Morocco,NaN,NaN
49502,2026-07-10,Spain,Belgium,NaN,NaN
49503,2026-07-11,Norway,England,NaN,NaN


In [72]:
away = df[["date", "home_team", "away_team", "home_score", "away_score"]].copy()
away.columns = ["date", "opponent", "team", "opponent_score", "goals_scored"]
away

,date,opponent,team,opponent_score,goals_scored
0,1872-11-30,Scotland,England,0.0,0.0
1,1873-03-08,England,Scotland,4.0,2.0
2,1874-03-07,Scotland,England,2.0,1.0
3,1875-03-06,England,Scotland,2.0,2.0
4,1876-03-04,Scotland,England,3.0,0.0
...,...,...,...,...,...
49500,2026-07-07,Switzerland,Colombia,0.0,0.0
49501,2026-07-09,France,Morocco,NaN,NaN
49502,2026-07-10,Spain,Belgium,NaN,NaN
49503,2026-07-11,Norway,England,NaN,NaN


In [ ]:
columns = list(away.columns)
idxTeam , idxOpp = columns.index("team"), columns.index("opponent")
idxHomeScore, idxOppScore = columns.index("goals_scored"), columns.index("opponent_score")
columns[idxTeam] , columns[idxOpp] = columns[idxOpp] , columns[idxTeam]
columns[idxHomeScore], columns[idxOppScore] = columns[idxOppScore], columns[idxHomeScore]
away = away[columns]

,date,team,opponent,goals_scored,opponent_score
0,1872-11-30,England,Scotland,0.0,0.0
1,1873-03-08,Scotland,England,2.0,4.0
2,1874-03-07,England,Scotland,1.0,2.0
3,1875-03-06,Scotland,England,2.0,2.0
4,1876-03-04,England,Scotland,0.0,3.0
...,...,...,...,...,...
49500,2026-07-07,Colombia,Switzerland,0.0,0.0
49501,2026-07-09,Morocco,France,NaN,NaN
49502,2026-07-10,Belgium,Spain,NaN,NaN
49503,2026-07-11,England,Norway,NaN,NaN


In [76]:
result = pd.concat([home, away], axis = 0)
result = result.sort_values(by = ["team", "date"])
result = result.reset_index(drop=True)

In [77]:
avg_goal = result.groupby('team')['goals_scored'].rolling(window = 5).mean().shift()
avg_goal = avg_goal.reset_index(drop=True)
result["avg_goal_scored"] = avg_goal
result

,date,team,opponent,goals_scored,opponent_score,avg_goal_scored
0,2012-09-25,Abkhazia,Artsakh,1.0,1.0,NaN
1,2012-10-21,Abkhazia,Artsakh,0.0,3.0,NaN
2,2013-09-23,Abkhazia,South Ossetia,3.0,0.0,NaN
3,2014-06-01,Abkhazia,Occitania,1.0,1.0,NaN
4,2014-06-02,Abkhazia,Sápmi,2.0,1.0,NaN
...,...,...,...,...,...,...
99005,2017-06-29,Åland Islands,Ynys Môn,2.0,0.0,0.8
99006,2023-07-09,Åland Islands,Isle of Wight,0.0,2.0,1.0
99007,2023-07-10,Åland Islands,Guernsey,1.0,4.0,1.0
99008,2023-07-11,Åland Islands,Western Isles,0.0,2.0,1.0
